# 01 · Data cleaning and construction of the analysis panel

**Project:** Red meat availability, hospital capacity, and colorectal cancer (CRC) in the United States, 1999–2020.

This notebook documents every step that turns the raw downloads in `original_datasets/` into the tidy state-year panel used in the report. Raw files are never modified.

Inputs
- `original_datasets/CDC_cancer/` — CDC WONDER U.S. Cancer Statistics (D205) XML exports for CRC, all ages and ages 20–49.
- `original_datasets/USDA_meat/mtpcc.csv` — USDA ERS per capita meat availability (carcass weight).
- `original_datasets/HCUP_SID/` — HCUP SID file-composition page (HTML, 1997–2010) and annual hospital-count reports (PDF, 2011–2020).
- `original_datasets/CDC_BRFSS/` — BRFSS state prevalence exports for smoking, obesity, and health insurance.

Outputs (in `outputs/cleaned/`)
- `crc_state_year_1999_2020.csv`
- `crc_state_early_1999_2020.csv`
- `crc_national_1999_2020.csv`
- `crc_national_ageband_1999_2020.csv`
- `hcup_sid_hospitals_1999_2020.csv`
- `brfss_state_covariates_1999_2020.csv`
- `usda_meat_per_capita_1909_2021.csv`
- `analysis_panel_1999_2020.csv`


In [1]:
import os, sys, tempfile
from pathlib import Path


def _find_project_root(start=None):
    """Walk up from the current directory until we find the repo root."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "data_prep.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open this notebook from inside the "
        "repository (the folder that contains src/, notebooks/, and outputs/)."
    )


ROOT = _find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mpl"))
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
print(f"Project root: {ROOT.name}")

Project root: DS-5003-group-4-Project


## 1. HCUP SID hospital counts

The HCUP page reports, for each year, the number of community hospitals in the SID, hospitals in the SID that are not community hospitals, and community hospitals not in the SID. For 2011–2020 the annual PDF reports give the total number of hospitals in the SID (`DSHOSPID`). We define **hospitals_sid = community hospitals in the SID + non-community hospitals in the SID** for 1997–2010, and use the reported SID total for 2011–2020. This is a consistent "hospitals participating in the SID" measure, not a count of every hospital in the state.

In [2]:
from src import data_prep as dp

hcup = dp.build_hcup()
print(f"Rows: {len(hcup):,}; states: {hcup.state_abbr.nunique()}; years: {hcup.year.min()}-{hcup.year.max()}")
hcup.groupby("year").size().rename("states_reporting").to_frame().T

Rows: 597; states: 36; years: 1999-2020


year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
states_reporting,23,23,23,23,23,24,24,25,25,26,26,26,29,27,26,30,30,31,31,33,34,35


In [3]:
hcup.head(8)

,year,state_abbr,hospitals_community_sid,hospitals_noncommunity_sid,hospitals_sid,hospitals_not_in_sid,source,state
0,1999,AZ,59.0,1.0,60.0,3.0,HCUP HTML table,Arizona
1,1999,CA,391.0,41.0,432.0,6.0,HCUP HTML table,California
2,1999,CO,66.0,3.0,69.0,1.0,HCUP HTML table,Colorado
3,1999,FL,193.0,24.0,217.0,12.0,HCUP HTML table,Florida
4,1999,HI,19.0,1.0,20.0,4.0,HCUP HTML table,Hawaii
5,1999,IA,115.0,1.0,116.0,0.0,HCUP HTML table,Iowa
6,1999,KY,NaN,NaN,NaN,NaN,HCUP HTML table,Kentucky
7,1999,MA,68.0,2.0,70.0,11.0,HCUP HTML table,Massachusetts


In [4]:
# Which states participate, and in which years?
coverage = (hcup.assign(observed=1)
            .pivot_table(index="state_abbr", columns="year", values="observed", aggfunc="max")
            .reindex(columns=range(1999, 2021)))
coverage = coverage.reindex(sorted(coverage.index))
coverage

year,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
state_abbr,,,,,,,,,,,,,,,,,,,,,,
AK,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0
AR,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
AZ,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
CA,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,NaN,NaN,NaN,1.0,1.0,1.0
CO,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
DC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
DE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0
FL,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
GA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


The SID is an **unbalanced panel**: 23 states reported in 1999 and 35 in 2020, and 15 states never appear. This is a structural feature of HCUP, not missing data that can be imputed safely. The report therefore uses hospital counts descriptively and restricts the hospital-density model to observed state-years.

## 2. CDC WONDER colorectal cancer incidence

The WONDER request groups by year and state, filters to colon and rectum (site codes `21041-21052`), and requests counts, populations, crude rates, and age-adjusted rates (2000 U.S. standard population). A second request returns state-year counts for adults aged 20–49 (early-onset CRC). A third returns national counts by 5-year age group.

In [5]:
crc = dp.parse_crc_state_year()
early = dp.parse_crc_state_early()
nat = dp.parse_crc_national()
age5, ageband = dp.parse_crc_national_age()

print("State-year, all ages:", crc.shape)
print("State-year, ages 20-49:", early.shape)
print("National years:", nat.shape)
print("National age bands:", ageband.shape)
crc.head()

State-year, all ages: (1116, 11)
State-year, ages 20-49: (1112, 6)
National years: (22, 5)
National age bands: (66, 5)


,year,state,state_fips,crc_cases,population,crc_age_adj_rate,crc_ci_low,crc_ci_high,crc_rate_se,crc_crude_rate,state_abbr
0,1999,Alabama,01,2250.0,4430141.0,49.72,47.68,51.82,1.05,50.79,AL
1,2000,Alabama,01,2305.0,4452173.0,50.48,48.43,52.58,1.05,51.77,AL
2,2001,Alabama,01,2337.0,4467634.0,50.57,48.54,52.67,1.05,52.31,AL
3,2002,Alabama,01,2356.0,4480089.0,50.36,48.34,52.44,1.04,52.59,AL
4,2003,Alabama,01,2376.0,4503491.0,50.07,48.07,52.14,1.03,52.76,AL


In [6]:
early.head()

,year,state,crc_early_cases,crc_early_population,crc_early_rate,state_abbr
0,1999,Alabama,222.0,1911329.0,11.61,AL
1,2000,Alabama,239.0,1910357.0,12.51,AL
2,2001,Alabama,237.0,1909040.0,12.41,AL
3,2002,Alabama,263.0,1903618.0,13.82,AL
4,2003,Alabama,230.0,1902680.0,12.09,AL


In [7]:
nat.head()

,year,crc_cases,population,crc_age_adj_rate,crc_crude_rate
0,1999,151469.0,275461348.0,56.00,54.99
1,2000,153175.0,278558214.0,55.74,54.99
2,2001,153645.0,282115961.0,54.95,54.46
3,2002,152446.0,284766512.0,53.61,53.53
4,2003,153916.0,290107933.0,52.66,53.05


In [8]:
ageband.pivot(index="year", columns="age_band", values="crc_crude_rate").round(2).head()

age_band,20-49,50-64,65+
year,,,
1999,9.56,85.33,307.02
2000,10.01,85.74,303.55
2001,10.17,84.18,298.98
2002,10.40,83.07,290.14
2003,10.57,82.96,282.96


## 3. USDA ERS meat availability

Per capita availability is a **disappearance** measure (production + imports + beginning stocks − exports − ending stocks, divided by population). It is not a direct measure of individual intake, but it is the only consistent annual U.S. series back to 1909. We use carcass-weight red meat and poultry.

In [9]:
meat = dp.build_meat()
meat[meat.year.between(1999, 2020)].head()

,year,red_meat_lb_pc,poultry_lb_pc,total_meat_lb_pc
90,1999,165.866351,106.497151,272.363502
91,2000,164.046065,107.393175,271.439240
92,2001,160.832340,107.212498,268.044838
93,2002,164.560115,112.060054,276.620169
94,2003,161.061511,113.076144,274.137655


In [10]:
meat[meat.year.between(1999, 2020)].set_index("year")[["red_meat_lb_pc", "poultry_lb_pc"]].tail()

,red_meat_lb_pc,poultry_lb_pc
year,,
2016,144.497445,121.573273
2017,146.718949,122.862129
2018,148.196530,124.433911
2019,151.219085,127.305530
2020,151.176607,128.451661


## 4. BRFSS state covariates

Smoking, obesity, and lack of health insurance are pulled from the CDC BRFSS prevalence datasets. We keep the `Overall` breakout, use crude prevalence, and prefer the modern variable definitions (`_RFSMOK3`, `_BMI5CAT`, `HLTHPLN1`) when both old and new versions exist. BRFSS changed its weighting methodology in 2011, so these covariates should be read as broad state-level context rather than a consistent clinical series.

In [11]:
brfss = dp.build_brfss_panel()
print(brfss.shape)
brfss.describe().round(1)

(1120, 5)


,year,smoking_pct,obesity_pct,uninsured_pct
count,1120.0,1120.0,1069.0,1120.0
mean,2009.5,19.7,26.7,13.7
std,6.3,4.0,5.0,4.5
min,1999.0,7.9,12.3,4.3
25%,2004.0,16.9,23.1,10.3
50%,2009.5,19.8,26.6,13.0
75%,2015.0,22.4,30.3,16.9
max,2020.0,32.6,40.8,30.6


In [12]:
brfss[brfss.state_abbr.isin(["CA", "TX", "WV"]) & brfss.year.isin([1999, 2010, 2020])]

,state_abbr,year,smoking_pct,obesity_pct,uninsured_pct
88,CA,1999,18.7,18.7,19.3
99,CA,2010,12.1,24.7,17.8
109,CA,2020,8.9,30.2,10.7
944,TX,1999,22.4,21.6,22.6
955,TX,2010,15.8,31.7,23.1
965,TX,2020,13.2,35.8,24.5
1076,WV,1999,27.1,24.6,18.1
1087,WV,2010,26.8,32.9,17.5
1097,WV,2020,22.6,39.1,8.0


## 5. Merge into the state-year analysis panel

The panel starts from the 51-state CRC file (50 states + DC), then left-joins HCUP, BRFSS, and national meat availability. Lags of meat availability (5, 10, 15, 20 years) are included because CRC develops slowly. Missing values are **retained** and handled inside each model pipeline; no outcome or exposure is imputed at this stage.

In [13]:
panel = dp.build_panel()
print(panel.shape)
print(panel.columns.tolist())
panel.head()

(1116, 39)
['year', 'state', 'state_fips', 'crc_cases', 'population', 'crc_age_adj_rate', 'crc_ci_low', 'crc_ci_high', 'crc_rate_se', 'crc_crude_rate', 'state_abbr', 'hospitals_sid', 'source', 'smoking_pct', 'obesity_pct', 'uninsured_pct', 'crc_early_cases', 'crc_early_population', 'crc_early_rate', 'region', 'red_meat_lb_pc', 'poultry_lb_pc', 'total_meat_lb_pc', 'red_meat_lag5', 'poultry_lag5', 'red_meat_lag10', 'poultry_lag10', 'red_meat_lag15', 'poultry_lag15', 'red_meat_lag20', 'poultry_lag20', 'hospitals_per100k', 'log_population', 'crc_rate_baseline', 'crc_rate_change', 'crc_early_rate_baseline', 'crc_early_change', 'hospitals_per100k_baseline', 'hospitals_per100k_change']


,year,state,state_fips,crc_cases,population,crc_age_adj_rate,crc_ci_low,crc_ci_high,crc_rate_se,crc_crude_rate,state_abbr,hospitals_sid,source,smoking_pct,obesity_pct,uninsured_pct,crc_early_cases,crc_early_population,crc_early_rate,region,red_meat_lb_pc,poultry_lb_pc,total_meat_lb_pc,red_meat_lag5,poultry_lag5,red_meat_lag10,poultry_lag10,red_meat_lag15,poultry_lag15,red_meat_lag20,poultry_lag20,hospitals_per100k,log_population,crc_rate_baseline,crc_rate_change,crc_early_rate_baseline,crc_early_change,hospitals_per100k_baseline,hospitals_per100k_change
0,1999,Alaska,02,209.0,624779.0,62.00,53.10,71.82,4.72,33.45,AK,NaN,NaN,27.3,20.4,23.3,22.0,298888.0,7.36,West,165.866351,106.497151,272.363502,164.784957,97.255537,167.062489,83.438559,175.120595,65.511445,177.522906,57.517733,NaN,13.345153,62.0,0.00,7.36,0.00,4.582272,NaN
1,2000,Alaska,02,232.0,627963.0,66.60,57.57,76.52,4.78,36.94,AK,NaN,NaN,25.0,21.0,18.1,31.0,296942.0,10.44,West,164.046065,107.393175,271.439240,164.379924,96.867715,162.298067,87.787074,176.650072,67.889221,179.886354,58.659120,NaN,13.350237,62.0,4.60,7.36,3.08,4.582272,NaN
2,2001,Alaska,02,239.0,633714.0,64.00,55.49,73.33,4.50,37.71,AK,NaN,NaN,26.2,22.1,18.8,32.0,295883.0,10.82,West,160.832340,107.212498,268.044838,160.350552,98.533008,161.772999,91.043436,173.961861,70.982824,177.626833,60.951375,NaN,13.359353,62.0,2.00,7.36,3.46,4.582272,NaN
3,2002,Alaska,02,215.0,642337.0,55.45,47.69,64.00,4.11,33.47,AK,NaN,NaN,29.3,23.4,19.1,25.0,296406.0,8.43,West,164.560115,112.060054,276.620169,157.413337,100.345351,164.275236,94.134178,169.838997,76.664054,170.099882,62.047423,NaN,13.372868,62.0,-6.55,7.36,1.07,4.582272,NaN
4,2003,Alaska,02,226.0,648414.0,53.78,46.41,61.90,3.90,34.85,AK,NaN,NaN,26.2,NaN,17.5,32.0,295811.0,10.82,West,161.061511,113.076144,274.137655,163.469324,101.394023,161.088353,95.819042,173.071072,79.474481,175.707069,63.613682,NaN,13.382285,62.0,-8.22,7.36,3.46,4.582272,NaN


In [14]:
missing = panel.isna().sum().sort_values(ascending=False)
missing[missing > 0].to_frame("missing_rows")

,missing_rows
hospitals_per100k_change,533
hospitals_per100k,533
hospitals_sid,533
source,519
hospitals_per100k_baseline,330
obesity_pct,53
crc_early_change,4
crc_early_rate,4
crc_early_population,4
crc_early_cases,4


In [15]:
panel.groupby("year").agg(
    states=("state_abbr", "nunique"),
    with_hospitals=("hospitals_per100k", lambda s: s.notna().sum()),
    median_crc=("crc_age_adj_rate", "median"),
    median_early=("crc_early_rate", "median"),
).round(2)

,states,with_hospitals,median_crc,median_early
year,,,,
1999,49,16,55.82,9.76
2000,49,19,55.30,10.14
2001,50,21,55.74,10.32
2002,50,22,53.60,11.01
2003,51,23,53.67,10.86
2004,51,24,51.29,11.21
2005,51,24,50.50,10.99
2006,51,25,48.91,11.58
2007,51,25,47.08,11.78


## 6. Save the cleaned files and a data dictionary

In [16]:
outputs = {
    "crc_state_year_1999_2020.csv": crc,
    "crc_state_early_1999_2020.csv": early,
    "crc_national_1999_2020.csv": nat,
    "crc_national_ageband_1999_2020.csv": ageband,
    "hcup_sid_hospitals_1999_2020.csv": hcup,
    "brfss_state_covariates_1999_2020.csv": brfss,
    "usda_meat_per_capita_1909_2021.csv": meat,
    "analysis_panel_1999_2020.csv": panel,
}
for name, frame in outputs.items():
    frame.to_csv(dp.CLEAN / name, index=False)
    print(f"wrote {name:42s} {frame.shape}")

wrote crc_state_year_1999_2020.csv               (1116, 11)
wrote crc_state_early_1999_2020.csv              (1112, 6)
wrote crc_national_1999_2020.csv                 (22, 5)
wrote crc_national_ageband_1999_2020.csv         (66, 5)
wrote hcup_sid_hospitals_1999_2020.csv           (597, 8)
wrote brfss_state_covariates_1999_2020.csv       (1120, 5)
wrote usda_meat_per_capita_1909_2021.csv         (113, 4)
wrote analysis_panel_1999_2020.csv               (1116, 39)


In [17]:
dictionary = pd.DataFrame([
    ("year", "Calendar year", "All sources"),
    ("state_abbr", "Two-letter USPS state code", "Derived"),
    ("crc_cases", "New invasive colon/rectum cancer cases, all ages", "CDC WONDER USCS D205"),
    ("population", "State population used as WONDER denominator", "CDC WONDER USCS D205"),
    ("crc_age_adj_rate", "Age-adjusted CRC incidence per 100,000 (2000 US standard)", "CDC WONDER USCS D205"),
    ("crc_crude_rate", "Crude CRC incidence per 100,000, all ages", "CDC WONDER USCS D205"),
    ("crc_early_cases", "New CRC cases, ages 20-49", "CDC WONDER USCS D205"),
    ("crc_early_rate", "Crude CRC incidence per 100,000, ages 20-49", "CDC WONDER USCS D205"),
    ("hospitals_sid", "Hospitals participating in the HCUP SID", "HCUP SID file composition"),
    ("hospitals_per100k", "hospitals_sid / population x 100,000", "Derived"),
    ("red_meat_lb_pc", "Red meat availability, lb per person per year (carcass weight)", "USDA ERS"),
    ("poultry_lb_pc", "Poultry availability, lb per person per year (carcass weight)", "USDA ERS"),
    ("red_meat_lag10", "Red meat availability 10 years earlier", "Derived"),
    ("smoking_pct", "Adults who currently smoke, crude %", "CDC BRFSS"),
    ("obesity_pct", "Adults with BMI >= 30, crude %", "CDC BRFSS"),
    ("uninsured_pct", "Adults without health insurance, crude %", "CDC BRFSS"),
    ("region", "Census region", "Derived"),
    ("crc_early_rate_baseline", "State's first observed early-onset rate (usually 1999)", "Derived"),
], columns=["column", "description", "source"])
dictionary.to_csv(dp.CLEAN / "data_dictionary.csv", index=False)
dictionary

,column,description,source
0,year,Calendar year,All sources
1,state_abbr,Two-letter USPS state code,Derived
2,crc_cases,"New invasive colon/rectum cancer cases, all ages",CDC WONDER USCS D205
3,population,State population used as WONDER denominator,CDC WONDER USCS D205
4,crc_age_adj_rate,"Age-adjusted CRC incidence per 100,000 (2000 U...",CDC WONDER USCS D205
5,crc_crude_rate,"Crude CRC incidence per 100,000, all ages",CDC WONDER USCS D205
6,crc_early_cases,"New CRC cases, ages 20-49",CDC WONDER USCS D205
7,crc_early_rate,"Crude CRC incidence per 100,000, ages 20-49",CDC WONDER USCS D205
8,hospitals_sid,Hospitals participating in the HCUP SID,HCUP SID file composition
9,hospitals_per100k,"hospitals_sid / population x 100,000",Derived


## 7. Validation checks

The checks below verify internal consistency: rates equal counts / population × 100,000; the national age-adjusted series is in a plausible range; and the early-onset series is a strict subset of the all-age counts.

In [18]:
check = crc.assign(recomputed=crc.crc_cases / crc["population"] * 100_000)
check["abs_diff"] = (check.recomputed - check.crc_crude_rate).abs()
print("Max crude-rate recomputation error:", round(check.abs_diff.max(), 4))
print("Rows with >0.1 error:", (check.abs_diff > 0.1).sum())

early_check = early.merge(crc[["year", "state_abbr", "crc_cases"]], on=["year", "state_abbr"], how="left")
print("Early-onset rows exceeding all-age counts:", (early_check.crc_early_cases > early_check.crc_cases).sum())
print("National 1999 / 2020 age-adjusted CRC:", nat.crc_age_adj_rate.iloc[0], nat.crc_age_adj_rate.iloc[-1])

Max crude-rate recomputation error: 0.005
Rows with >0.1 error: 0
Early-onset rows exceeding all-age counts: 0
National 1999 / 2020 age-adjusted CRC: 56.0 33.82


All checks pass (or are documented) and the cleaned panel is ready for the exploratory and modeling notebooks.